# RAG Pipeline for Privacy Policy

### Install the necessary libraries


In [ ]:
#%pip install requests
#%pip install beautifulsoup4
#%pip install sentence-transformers
#%pip install faiss-cpu
#%pip install transformers
#%pip install torch
#%pip install pandas
#%pip install numpy
#%pip install streamlit
#%pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import the libraries

In [3]:
import requests
from bs4 import BeautifulSoup

import numpy as np
import pandas as pd

import faiss

from pypdf import PdfReader

from io import BytesIO
from sentence_transformers import SentenceTransformer

from transformers import pipeline

import streamlit as st

m:\Coding\RAG_Pipeline_PrivacyPolicy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Extract the text if Privacy Policy is given as URL

In [4]:
def extract_text_from_url(url):
    """
    Downloads webpage and extracts visible text.
    """

    response = requests.get(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0"
            )
        },
        timeout=20
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Remove unwanted elements
    for tag in soup([
        "script",
        "style",
        "header",
        "footer",
        "nav"
    ]):
        tag.decompose()

    text = soup.get_text(
        separator="\n",
        strip=True
    )

    return text

### Extract the text if Privacy Policy is given with .pdf extension

In [5]:
def extract_text_from_pdf(uploaded_file):
    """
    Extract text from uploaded PDF.
    """

    pdf = PdfReader(
        BytesIO(uploaded_file.read())
    )

    text = ""

    for page in pdf.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

In [6]:
import re


def clean_text(text):

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Remove multiple newlines
    text = re.sub(
        r"\n+",
        "\n",
        text
    )

    # Remove tabs
    text = text.replace(
        "\t",
        " "
    )

    return text.strip()

In [7]:
def load_policy(source_type,
                source):

    if source_type == "url":

        raw_text = extract_text_from_url(
            source
        )

    elif source_type == "pdf":

        raw_text = extract_text_from_pdf(
            source
        )

    else:

        raise ValueError(
            "Unsupported source type"
        )

    cleaned_text = clean_text(
        raw_text
    )

    return cleaned_text

In [8]:
policy_text = load_policy(
    source_type="url",
    source="https://policies.google.com/privacy?hl=en-US"
)

print(policy_text[:1000])

Privacy Policy – Privacy & Terms – Google Skip to main content Privacy & Terms Privacy & Terms Privacy & Terms Overview Privacy Policy Terms of Service Technologies FAQ Google Account Privacy Policy Introduction Information Google collects Why Google collects data Your privacy controls Sharing your information Keeping your information secure Exporting & deleting your information Retaining your information Compliance & cooperation with regulators European requirements About this policy Related privacy practices Data transfer frameworks Key terms Partners Updates Google Privacy Policy When you use our services, you’re trusting us with your information. We understand this is a big responsibility and work hard to protect your information and put you in control. This Privacy Policy is meant to help you understand what information we collect, why we collect it, and how you can update, manage, export, and delete your information. If European Union or United Kingdom data protection law applies

In [9]:
print(
    f"Characters: {len(policy_text)}"
)

print(
    f"Words: {len(policy_text.split())}"
)

Characters: 89751
Words: 14439


In [10]:
def chunk_text(
    text,
    chunk_size=250,
    overlap=50
):
    """
    Split text into overlapping word chunks.

    Parameters:
        text (str)
        chunk_size (int)
        overlap (int)

    Returns:
        List[str]
    """

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        chunks.append(chunk)

        start += (
            chunk_size - overlap
        )

    return chunks

In [11]:
chunks = chunk_text(
    policy_text,
    chunk_size=250,
    overlap=50
)

print(
    f"Total Chunks: {len(chunks)}"
)

Total Chunks: 73


In [12]:
for i in range(3):

    print(
        f"\nChunk {i+1}"
    )

    print(
        f"Words: {len(chunks[i].split())}"
    )


Chunk 1
Words: 250

Chunk 2
Words: 250

Chunk 3
Words: 250


In [13]:
def create_chunk_metadata(
    chunks
):

    chunk_data = []

    for idx, chunk in enumerate(
        chunks
    ):

        chunk_data.append(
            {
                "chunk_id": idx,
                "text": chunk
            }
        )

    return chunk_data


In [14]:
for chunk in chunks[:5]:

    print("=" * 50)

    print(chunk[:500])

    print("\n")

Privacy Policy – Privacy & Terms – Google Skip to main content Privacy & Terms Privacy & Terms Privacy & Terms Overview Privacy Policy Terms of Service Technologies FAQ Google Account Privacy Policy Introduction Information Google collects Why Google collects data Your privacy controls Sharing your information Keeping your information secure Exporting & deleting your information Retaining your information Compliance & cooperation with regulators European requirements About this policy Related pr


Information Google collects Why Google collects data Your privacy controls Sharing your information Keeping your information secure Exporting & deleting your information Retaining your information Compliance & cooperation with regulators European requirements About this policy Related privacy practices We build a range of services that help millions of people daily to explore and interact with the world in new ways. Our services include: Google apps, sites, and devices, like Search, YouTube, 

In [15]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1445.11it/s]


In [16]:
def generate_embeddings(
    chunks,
    model
):
    """
    Generate embeddings for all chunks.
    """

    embeddings = model.encode(
        chunks,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    return embeddings

In [17]:
embeddings = generate_embeddings(
    chunks,
    embedding_model
)

Batches: 100%|██████████| 3/3 [00:13<00:00,  4.45s/it]


In [18]:
print(embeddings.shape)

(73, 384)


In [19]:
print(embeddings[0][:10])

[-0.06676044 -0.00260836  0.02742872 -0.04394981  0.09156968  0.0495106
  0.09883881 -0.09844571 -0.01740689 -0.00611733]


In [20]:
def create_faiss_index(
    embeddings
):
    """
    Create FAISS index from embeddings.
    """

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(
        dimension
    )

    index.add(
        embeddings.astype(
            np.float32
        )
    )

    return index

In [21]:
index = create_faiss_index(
    embeddings
)

In [22]:
print(
    f"Vectors stored: {index.ntotal}"
)

Vectors stored: 73


In [23]:
question = (
    "Does this website share my data?"
)

query_embedding = (
    embedding_model.encode(
        [question],
        convert_to_numpy=True
    )
)

In [24]:
def retrieve_chunks(
    query,
    model,
    index,
    chunk_data,
    top_k=5
):
    # Generate query embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # FAISS requires float32
    query_embedding = query_embedding.astype("float32")

    # Search
    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:

        # Ignore invalid indices
        if idx < len(chunk_data):

            results.append({
                "chunk_id": chunk_data[idx]["chunk_id"],
                "text": chunk_data[idx]["text"]
            })

    return results

In [25]:
chunk_data = create_chunk_metadata(chunks)

print("chunk_data created")
print(len(chunk_data))

chunk_data created
73


In [26]:
results = retrieve_chunks(
    query="Does this website share my data?",
    model=embedding_model,
    index=index,
    chunk_data = chunk_data,
    top_k=5
)

for result in results:

    print("\n")
    print("="*80)

    print(result["chunk_id"])

    print(result["text"][:500])



71
before the activation. Learn more your activity on other sites and apps This activity might come from your use of Google services, like from syncing your account with Chrome or your visits to sites and apps that partner with Google. Many websites and apps partner with Google to improve their content and services. For example, a website might use our advertising services (like AdSense) or analytics tools (like Google Analytics), or it might embed other content (such as videos from YouTube). These


40
(like IP address, crash reports, and system activity). Information about your activity in our services , such as your search terms, Chrome browsing history you’ve synced with your Google Account, your views and interactions with content and ads, and your activity on third-party sites and apps that use our services. You can review and control activity data stored in your Google Account in My Activity . Location information , such as may be determined by GPS, IP address, and other data 

In [27]:
def build_context(
    retrieved_chunks
):

    context = ""

    for chunk in retrieved_chunks:

        context += (
            chunk["text"]
            + "\n\n"
        )

    return context

In [28]:
context = build_context(
    results
)

In [29]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_answer(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        # do_sample=True,          # use greedy decoding; set True + temperature if you want sampling
        do_sample=False,          # use greedy decoding; set True + temperature if you want sampling
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 892.85it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [30]:
def build_prompt(
    context,
    question
):

    prompt = f"""
You are a privacy policy assistant.

Answer ONLY using the information provided in the context.

If the answer is not found in the context, respond:
"Information not found in the policy."

Provide a concise answer.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [31]:
query = "Does this website share my data?"
prompt = build_prompt(context, query)

# Using the updated explicit generation function
response = generate_answer(
    prompt,
    max_new_tokens=150
)

# The generate_answer function already returns the decoded string, so we can directly assign it
answer = response
print(f"Answer: {answer}")

Answer: Information not found in the policy.


In [32]:
def ask_policy_question(question, model_embedding, index, chunk_data, llm_func):
    retrieved = retrieve_chunks(
        query=question,
        model=model_embedding,
        index=index,
        chunk_data=chunk_data,
        top_k=5
    )
    context = build_context(retrieved)
    prompt = build_prompt(context, question)

    model_response = llm_func(
        prompt,
        max_new_tokens=150
    )

    return model_response, retrieved

In [33]:
model_response, evidence = ask_policy_question(
    question="How long is user data retained?",
    model_embedding=embedding_model,
    index=index,
    chunk_data=chunk_data,
    llm_func=generate_answer
)

# Extract the answer from the full model response
answer = model_response
print(f"Answer: {answer}")

Answer: Information not found in the policy.
